# Sprint 4 Runner (Colab)

<!-- Cell 0: Judul notebook - intro Sprint 4 Plan v3 -->

Notebook khusus untuk menjalankan `Sprint 4 Plan v3` end-to-end dengan output log tampil penuh di setiap sel.

## 0) Settings

<!-- Cell 1: Header section Settings -->

Isi `REPO_URL` dengan repository kamu. Kalau repo private, pastikan token/Git auth sudah siap di Colab.

In [ ]:
# Cell 2: Core settings - REPO_URL, branch, path, stage toggles
REPO_URL = 'https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git'  # contoh: https://github.com/<user>/<repo>.git
REPO_BRANCH = 'feat/sprint4-anomaly-first-low-fpr'  # branch yang mau dipakai di Colab
PROJECT_NAME = 'nids-cnn-lstm-autoencoder'
DRIVE_ROOT = '/content/drive/MyDrive/nids-cnn-lstm-autoencoder'

# Raw dataset source (default mengikuti Sprint 3)
# Jika folder ini tidak ada, notebook fallback ke {DRIVE_ROOT}/data/raw
RAW_DRIVE_SOURCE = '/content/drive/MyDrive/nids-data/raw'

FORCE_RECLONE = False

# Stage toggles (set True/False sesuai kebutuhan)
RUN_STAGE0 = True
RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_STAGE3_4 = True
RUN_SUMMARIZE = True


In [ ]:
# Cell 3: Colab bootstrap + mount Google Drive
import os
import sys
import json
import time
import shlex
import shutil
import subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB =', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)


In [ ]:
# Cell 4: Clone atau update repo di /content + checkout branch
PROJECT_ROOT = Path('/content') / PROJECT_NAME

if FORCE_RECLONE and PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

if not PROJECT_ROOT.exists():
    if '<REPO_URL_HERE>' in REPO_URL:
        raise ValueError('Set REPO_URL dulu di cell Settings')
    cmd = ['git', 'clone', REPO_URL, str(PROJECT_ROOT)]
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)

os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('cwd =', Path.cwd())

# Always sync and checkout selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--all', '--prune'], check=False)

# Try checkout branch; if branch only exists on origin, create tracking local branch.
ret = subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_BRANCH], check=False)
if ret.returncode != 0:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '-b', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)

# Pull latest on selected branch
subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', REPO_BRANCH], check=False)

active_branch = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'branch', '--show-current'], text=True).strip()
print('[GIT] active branch =', active_branch)


In [ ]:
# Cell 5: Install dependencies (Colab-safe, skip reinstall numpy/tensorflow)
from pathlib import Path
import importlib.util

req = Path('requirements.txt').resolve()
if not req.exists():
    raise FileNotFoundError(f'requirements.txt not found at {req}')


def _run(cmd):
    print('[CMD]', ' '.join(cmd))
    subprocess.run(cmd, check=True)


if IN_COLAB:
    # IMPORTANT:
    # Colab sudah punya numpy/pandas/scikit/tensorflow yang saling kompatibel.
    # Reinstall paket-paket ini sering memicu ABI mismatch (numpy.dtype size changed).
    print('[INFO] Colab mode: skip reinstall scientific stack (numpy/pandas/scikit-learn/tensorflow).')

    # Install only lightweight utilities if missing.
    lightweight = [
        'pyyaml',
        'joblib',
        'seaborn',
    ]

    for pkg in lightweight:
        mod = 'yaml' if pkg == 'pyyaml' else pkg
        if importlib.util.find_spec(mod) is None:
            _run([sys.executable, '-m', 'pip', 'install', pkg])
        else:
            print(f'[INFO] {pkg} already available')

    print('\n[OK] Dependency step finished (Colab-safe mode).')
    print('[NOTE] Jika sebelumnya sempat install ulang numpy/tensorflow dan masih error, lakukan Factory reset runtime dulu.')
else:
    # Local/non-Colab: follow project requirements as usual.
    _run([sys.executable, '-m', 'pip', 'install', '-r', str(req)])


In [ ]:
# Cell 6: Link data/model/results sprint4 ke Drive (persist setelah disconnect)
os.chdir(PROJECT_ROOT)

raw_source = RAW_DRIVE_SOURCE if Path(RAW_DRIVE_SOURCE).exists() else f'{DRIVE_ROOT}/data/raw'
print(f'[RAW] using source: {raw_source}')

paths = [
    ('data/raw', raw_source),
    ('data/sprint4', f'{DRIVE_ROOT}/data/sprint4'),
    ('models/sprint4', f'{DRIVE_ROOT}/models/sprint4'),
    ('results/sprint4', f'{DRIVE_ROOT}/results/sprint4'),
]

for _, dst in paths:
    Path(dst).mkdir(parents=True, exist_ok=True)

for src, dst in paths:
    src_path = Path(src)
    if src_path.is_symlink() or src_path.exists():
        if src_path.is_symlink() or src_path.is_file():
            src_path.unlink()
        else:
            shutil.rmtree(src_path)
    src_path.parent.mkdir(parents=True, exist_ok=True)
    src_path.symlink_to(Path(dst), target_is_directory=True)
    print(f'[LINK] {src} -> {dst}')

print('[OK] Symlink setup complete')

# Fast preflight check
required_dirs = [
    Path('data/raw/CIC-IDS2017'),
    Path('data/raw/CSE-CIC-IDS2018'),
]
for d in required_dirs:
    print(f'[CHECK] {d}:', 'OK' if d.exists() else 'MISSING')


In [ ]:
# Cell 7: GPU check + run_cmd_stream helper (streaming output ke cell)
try:
    import tensorflow as tf
except Exception as e:
    print('[ERROR] TensorFlow import gagal:', repr(e))
    print('Kemungkinan besar environment ABI belum sinkron setelah pip install.')
    print('Solusi: Runtime > Restart runtime, lalu jalankan lagi dari cell GPU check ini.')
    raise

print('Python executable :', sys.executable)
print('TensorFlow        :', tf.__version__)
print('Built with CUDA   :', tf.test.is_built_with_cuda())
print('Built with GPU sup:', tf.test.is_built_with_gpu_support())
print('Physical GPU list :', tf.config.list_physical_devices('GPU'))
print('Logical GPU list  :', tf.config.list_logical_devices('GPU'))

# Maksimalkan penggunaan GPU: memory growth agar TF pakai memori dinamis
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass
if gpus:
    print('[GPU] Memory growth enabled -> training/eval akan memaksimalkan GPU.')


def _fmt_duration(seconds: float) -> str:
    sec = max(0, int(seconds))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h}h {m:02d}m {s:02d}s'
    if m > 0:
        return f'{m}m {s:02d}s'
    return f'{s}s'


def _normalize_cmd(cmd: str):
    parts = shlex.split(cmd, posix=False)
    if parts and parts[0].lower() == 'python':
        parts[0] = sys.executable
    return parts


def run_cmd_stream(title: str, cmd: str, log_to_file: bool = True):
    """Jalankan command dengan streaming output ke cell. Log lengkap disimpan ke file."""
    print(f'[RUN] {title}', flush=True)
    args = _normalize_cmd(cmd)
    print('[CMD]', ' '.join(args), flush=True)
    t0 = time.time()

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    proc = subprocess.Popen(
        args,
        cwd=str(PROJECT_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    log_path = None
    log_file = None
    if log_to_file:
        log_dir = PROJECT_ROOT / 'results' / 'sprint4' / 'runtime'
        log_dir.mkdir(parents=True, exist_ok=True)
        ts = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
        log_path = log_dir / f'log_{title.replace(" ", "_")}_{ts}.txt'
        log_file = open(log_path, 'w', encoding='utf-8')

    last_line = ''
    try:
        for line in proc.stdout or []:
            print(line, end='', flush=True)
            if log_file:
                log_file.write(line)
                log_file.flush()
            last_line = line.strip()
    finally:
        if log_file:
            log_file.close()
            print(f'\n[LOG] Full log saved to: {log_path}', flush=True)

    ret = proc.wait()
    print(f'\n[EXIT CODE] {ret}', flush=True)
    if ret != 0:
        raise RuntimeError(f"{title} failed (exit={ret}). Last line: {last_line}")

    print(f'[DONE] {title} in {_fmt_duration(time.time() - t0)}', flush=True)


def run_stage(stage_name: str):
    run_cmd_stream(
        title=f'Sprint4 {stage_name}',
        cmd=f'python scripts/sprint4/research_runner.py --stage-names {stage_name} --skip-existing',
    )


## 1) Optional Dry-Run

<!-- Cell 8: Header dry-run - validasi tanpa eksekusi real -->

Validasi command tanpa eksekusi real training/eval.

In [ ]:
# Cell 9: Execute dry-run stage0 (validasi command tanpa training/eval real)
run_cmd_stream(
    title='Sprint4 Dry-Run Stage0',
    cmd='python scripts/sprint4/research_runner.py --dry-run --stage-names stage0 --no-summarize',
)


## 2) Execute Sprint 4 Stages

<!-- Cell 10: Header execute stages -->

In [ ]:
# Cell 11: Execute stage0 (repro + strict contract runs)
if RUN_STAGE0:
    run_stage('stage0')
else:
    print('[SKIP] stage0')


In [ ]:
# Cell 12: Execute stage1 (6 preprocessing variant runs)
if RUN_STAGE1:
    run_stage('stage1')
else:
    print('[SKIP] stage1')


In [ ]:
# Cell 13: Execute stage2 (8 model architecture variant runs)
if RUN_STAGE2:
    run_stage('stage2')
else:
    print('[SKIP] stage2')


In [11]:
# Cell 14: Execute stage3 + stage4 (threshold variants + seed candidates)
if RUN_STAGE3_4:
    run_cmd_stream(
        title='Sprint4 stage3+stage4',
        cmd='python scripts/sprint4/research_runner.py --stage-names stage3,stage4 --skip-existing',
    )
else:
    print('[SKIP] stage3,stage4')


[RUN] Sprint4 stage3+stage4
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --stage-names stage3,stage4 --skip-existing
[RUN] s4_t01_source_percentile_p93 | stage=stage3 | stages=['eval'] | tag=s4_t01_source_percentile_p93
[CMD] /usr/bin/python3 scripts/sprint4/eval.py --config /content/nids-cnn-lstm-autoencoder/research/sprint4/generated_configs/s4_t01_source_percentile_p93.yaml --model /content/sprint4_lock/s4_t01_source_percentile_p93/best_model.keras --tag s4_t01_source_percentile_p93
2026-02-28 16:49:51.740223: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772297391.760846   82683 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772297391.768146   82683 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to regis

In [12]:
# Cell 15: Execute summarize-only (aggregate hasil, gate decision, report)
if RUN_SUMMARIZE:
    run_cmd_stream(
        title='Sprint4 summarize-only',
        cmd='python scripts/sprint4/research_runner.py --summarize-only',
    )
else:
    print('[SKIP] summarize-only')


[RUN] Sprint4 summarize-only
[CMD] /usr/bin/python3 scripts/sprint4/research_runner.py --summarize-only
[DONE] summary: /content/nids-cnn-lstm-autoencoder/results/sprint4/summary.csv
[DONE] report: /content/nids-cnn-lstm-autoencoder/docs/sprint4/RESEARCH_REPORT_CSE_F1_SPRINT4.md
[DONE] gate: /content/nids-cnn-lstm-autoencoder/results/sprint4/gate_decision.json

[LOG] Full log saved to: /content/nids-cnn-lstm-autoencoder/results/sprint4/runtime/log_Sprint4_summarize-only_20260228_184031.txt

[EXIT CODE] 0
[DONE] Sprint4 summarize-only in 1s


## 3) Show Outputs (Summary, Gate, Report)

<!-- Cell 16: Header show outputs -->

In [13]:
# Cell 17: Tampilkan summary.csv, gate_decision.json, report markdown
import pandas as pd
from IPython.display import display, Markdown

summary_path = PROJECT_ROOT / 'results/sprint4/summary.csv'
gate_path = PROJECT_ROOT / 'results/sprint4/gate_decision.json'
report_path = PROJECT_ROOT / 'docs/sprint4/RESEARCH_REPORT_CSE_F1_SPRINT4.md'
status_path = PROJECT_ROOT / 'results/sprint4/runtime/run_status.json'

print('summary_path =', summary_path)
print('gate_path    =', gate_path)
print('report_path  =', report_path)
print('status_path  =', status_path)

if summary_path.exists():
    df = pd.read_csv(summary_path)
    print('\n[SUMMARY HEAD]')
    display(df.head(30))
    print('\n[STATUS COUNTS]')
    if 'terminal_status' in df.columns:
        display(df['terminal_status'].value_counts(dropna=False))
else:
    print('[WARN] summary.csv not found')

if gate_path.exists():
    gate = json.loads(gate_path.read_text(encoding='utf-8'))
    print('\n[GATE DECISION]')
    print(json.dumps(gate, indent=2))
else:
    print('[WARN] gate_decision.json not found')

if report_path.exists():
    print('\n[REPORT PREVIEW]')
    txt = report_path.read_text(encoding='utf-8')
    display(Markdown(txt[:8000]))
else:
    print('[WARN] report markdown not found')

if status_path.exists():
    print('\n[RUNTIME STATUS JSON]')
    print(status_path.read_text(encoding='utf-8')[:8000])


summary_path = /content/nids-cnn-lstm-autoencoder/results/sprint4/summary.csv
gate_path    = /content/nids-cnn-lstm-autoencoder/results/sprint4/gate_decision.json
report_path  = /content/nids-cnn-lstm-autoencoder/docs/sprint4/RESEARCH_REPORT_CSE_F1_SPRINT4.md
status_path  = /content/nids-cnn-lstm-autoencoder/results/sprint4/runtime/run_status.json

[SUMMARY HEAD]


,run_id,stage_name,profile,model_variant,active,condition,stages,tag,terminal_status,retry_count,...,cse_prec,cse_rec,cse_f1,cse_fpr,cse_auc,f1_gap,auc_gap,accuracy_gap,wall_time_sec,last_error
0,s4_00_repro_v4_full,stage0,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_00_repro_v4_full,success,0,...,0.920498,0.612420,0.735501,0.131320,NaN,0.059531,0.0,0.042528,1341.554662,NaN
1,s4_01_strict_contract_full,stage0,s4_hybrid_zero_shot_anomaly,hybrid,True,always,eval,s4_01_strict_contract_full,success,0,...,0.920498,0.612420,0.735501,0.131320,NaN,0.059531,0.0,0.042528,35.466293,NaN
2,s4_p01_robust_q995_clip20_corr090,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p01_robust_q995_clip20_corr090,success,0,...,0.894403,0.489922,0.633070,0.143575,NaN,0.160690,0.0,0.133133,27292.762990,NaN
3,s4_p02_robust_q997_clip20_corr090,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p02_robust_q997_clip20_corr090,success,0,...,0.792635,0.249893,0.379988,0.162275,NaN,0.422581,0.0,0.319248,5765.759168,NaN
4,s4_p03_quantile_q995_clip20_corr090,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p03_quantile_q995_clip20_corr090,success,0,...,0.925205,0.598594,0.726897,0.120116,NaN,0.069481,0.0,0.052405,7502.106961,NaN
5,s4_p04_robust_q995_clip15_corr085,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p04_robust_q995_clip15_corr085,success,0,...,0.777075,0.196009,0.313053,0.139574,NaN,0.496282,0.0,0.358783,11980.480413,NaN
6,s4_p05_robust_q995_clip20_corr095,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p05_robust_q995_clip20_corr095,success,0,...,0.766130,0.217375,0.338661,0.164709,NaN,0.455613,0.0,0.333797,5594.926673,NaN
7,s4_p06_robust_q995_clip20_win20_stride2,stage1,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"preprocess,train,eval",s4_p06_robust_q995_clip20_win20_stride2,success,0,...,0.759435,0.052190,0.097668,0.042282,NaN,0.665515,0.0,0.383976,672.474445,NaN
8,s4_m01_baseline_arch,stage2,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"train,eval",s4_m01_baseline_arch,success,0,...,0.909970,0.452966,0.604849,0.111240,NaN,0.182549,0.0,0.144120,1071.252572,NaN
9,s4_m02_dropout_030,stage2,s4_hybrid_zero_shot_anomaly,hybrid,True,always,"train,eval",s4_m02_dropout_030,success,0,...,0.931947,0.595101,0.726372,0.107866,NaN,0.062019,0.0,0.042553,1059.755638,NaN



[STATUS COUNTS]


,count
terminal_status,
success,23
failed_experiment,1



[GATE DECISION]
{
  "generated_at": "2026-02-28T18:40:33.309971",
  "gate_pass": true,
  "reason": "passed",
  "adaptive_recall_target": 0.4,
  "adaptive_target_reasoning": "max(0.4000, baseline(0.3442)+0.0500)",
  "best_stage3_run_id": "s4_t06_source_calib_guardrail_fpr010",
  "best_stage3_metrics": {
    "cse_recall": 0.6904132821426512,
    "cse_precision": 0.8632611733612956,
    "cse_f1": 0.7672224743546661,
    "cic_fpr": 0.09305118048841343,
    "threshold": 0.011529527604579926
  },
  "pivot_recommendation": null,
  "stage_validity": {
    "stage1": {
      "required": 5,
      "valid_count": 6,
      "passed": true,
      "status": "ok"
    },
    "stage2": {
      "required": 6,
      "valid_count": 7,
      "passed": true,
      "status": "ok"
    },
    "stage3": {
      "required": 5,
      "valid_count": 6,
      "passed": true,
      "status": "ok"
    },
    "stage4": {
      "required": 2,
      "valid_count": 2,
      "passed": true,
      "status": "ok"
    }
  },
 

# RESEARCH REPORT CSE F1 - SPRINT 4

- generated_at: 2026-02-28T18:40:33.313754
- source_summary: `/content/nids-cnn-lstm-autoencoder/results/sprint4/summary.csv`
- gate_decision: `/content/nids-cnn-lstm-autoencoder/results/sprint4/gate_decision.json`

## Ringkasan / Summary

- Objective: maximize CSE recall dengan guardrail CIC FPR rendah.
- Objective (EN): maximize CSE recall under low CIC-FPR guardrail.
- Gate pass: `True` (reason: `passed`).
- Adaptive recall target: `0.4` (max(0.4000, baseline(0.3442)+0.0500)).

## Stage Validity

| stage | required_valid_runs | valid_runs | status |
| --- | ---: | ---: | --- |
| stage1 | 5 | 6 | ok |
| stage2 | 6 | 7 | ok |
| stage3 | 5 | 6 | ok |
| stage4 | 2 | 2 | ok |

## Best Candidate Per Stage

| stage | run_id | mode | cse_recall | cse_precision | cse_f1 | cic_fpr |
| --- | --- | --- | ---: | ---: | ---: | ---: |
| stage0 | s4_00_repro_v4_full | hard | 0.6124 | 0.9205 | 0.7355 | 0.0555 |
| stage1 | s4_p03_quantile_q995_clip20_corr090 | hard | 0.5986 | 0.9252 | 0.7269 | 0.0304 |
| stage2 | s4_m06_loss_huber | hard | 0.6391 | 0.9308 | 0.7579 | 0.0323 |
| stage3 | s4_t06_source_calib_guardrail_fpr010 | hard | 0.6904 | 0.8633 | 0.7672 | 0.0931 |
| stage4 | s4_c02_best_seed1234_full | hard | 0.6892 | 0.8636 | 0.7666 | 0.0978 |

## Operational Notes

- ID Primary: Terminologi teknis tetap English untuk konsistensi.
- EN Mirror: Technical keywords are intentionally kept in English.

## Run Status

- total_runs: 24
- success: 23
- failed_experiment: 1
- failed_infra_exhausted: 0
- pending_or_skipped: 0



[RUNTIME STATUS JSON]
{
  "s4_p01_robust_q995_clip20_corr090": {
    "retry_count": 0,
    "terminal_status": "success",
    "last_error": "",
    "hash_violation": false,
    "wall_time_sec": 27292.762989997864,
    "attempt_logs": [
      {
        "ts": "2026-02-27T15:16:24.383200",
        "kind": "experiment",
        "error": "Command failed rc=1 after 1.52s: /usr/bin/python3 scripts/sprint4/preprocess.py --config /content/nids-cnn-lstm-autoencoder/research/sprint4/generated_configs/s4_p01_robust_q995_clip20_corr090.yaml"
      }
    ]
  },
  "s4_p02_robust_q997_clip20_corr090": {
    "retry_count": 0,
    "terminal_status": "success",
    "last_error": "",
    "hash_violation": false,
    "wall_time_sec": 5765.759167909622,
    "attempt_logs": [
      {
        "ts": "2026-02-27T15:16:25.928900",
        "kind": "experiment",
        "error": "Command failed rc=1 after 1.53s: /usr/bin/python3 scripts/sprint4/preprocess.py --config /content/nids-cnn-lstm-autoencoder/research/spr

## 4) Resume Guide

<!-- Cell 18: Panduan resume setelah runtime putus -->

Kalau runtime Colab putus:
1. Run lagi dari cell mount + clone/pull + symlink.
2. Jalankan stage berikutnya atau stage yang sama.
3. `--skip-existing` akan melanjutkan dari artifact yang sudah ada.